In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16

# Definindo os caminhos e parâmetros básicos
caminho_treino = './fer2013/train'
caminho_teste = './fer2013/test'
tamanho_lote = 32 # Quantas imagens a IA vê por vez
tamanho_imagem = (48, 48) # Tamanho padrão do FER2013

print("Carregando dados de treinamento...")
dados_treino = tf.keras.utils.image_dataset_from_directory(
    caminho_treino,
    color_mode="rgb", # A VGG16 exige imagens com 3 canais de cor (RGB)
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

print("\nCarregando dados de validação (teste)...")
dados_teste = tf.keras.utils.image_dataset_from_directory(
    caminho_teste,
    color_mode="rgb",
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

# Carrega a rede pré-treinada sem o "topo" (include_top=False)
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(48, 48, 3))

# Congela os pesos para não estragar o que ela já aprendeu
base_vgg.trainable = False

# Construindo o novo "topo" da rede
# Como a base_vgg está com trainable=False, apenas as camadas abaixo aprenderão agora.
modelo_vgg = models.Sequential([
    base_vgg,           # A base da VGG16 (extrator de características)
    layers.Flatten(),   # Transforma os mapas de características 2D em um vetor 1D
    layers.Dense(256, activation='relu'), # Camada densa para processar os padrões extraídos
    layers.Dropout(0.5), # Ajuda a prevenir o overfitting durante o treinamento
    layers.Dense(7, activation='softmax')  # Saída final para as 7 classes de emoções do FER2013
])

# Compilando o modelo
# Usamos o otimizador Adam, que é padrão para esse tipo de tarefa de classificação.
modelo_vgg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Resumo da Arquitetura
# Observe que a maioria dos parâmetros (da VGG) aparecerá como "Non-trainable params".
modelo_vgg.summary()

# Treinamento de Teste (3 épocas)
# Lembre-se: como as imagens foram carregadas em RGB, o processamento será um pouco mais lento que o baseline.
print("\nIniciando o treinamento com VGG16... Acompanhe a evolução da acurácia.")
historico = modelo_vgg.fit(
    dados_treino,
    validation_data=dados_teste,
    epochs=3
)